# CyberSec Lab_ Network Traffic Analysis + Vulnerability Scanning

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
#!/usr/bin/env python3

import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Try importing libraries (install if needed)
try:
    from scapy.all import *
    from scapy.layers.inet import IP, TCP, UDP, ICMP
except ImportError:
    print("⚠️  Installing scapy...")
    os.system("pip install -q scapy")
    from scapy.all import *

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ImportError:
    print("⚠️  Installing plotly...")
    os.system("pip install -q plotly")
    import plotly.express as px

try:
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
except ImportError:
    print("⚠️  Installing scikit-learn...")
    os.system("pip install -q scikit-learn")
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler


# ═══════════════════════════════════════════════════════════════════════════
# PART 1: NETWORK TRAFFIC ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════

def generate_sample_traffic(num_packets=1000, malicious_ratio=0.12):
    """
    Generate synthetic network traffic for testing.
    Mix of normal HTTP/DNS/ICMP and suspicious patterns.
    """
    print(f"🔄 Generating {num_packets} sample packets...\n")

    packets = []
    normal_ips = ['192.168.1.10', '192.168.1.11', '192.168.1.12', '10.0.0.5']
    suspicious_ips = ['185.220.101.45', '45.142.212.61', '23.129.64.131']  # Known Tor exit nodes
    common_ports = [80, 443, 53, 22]
    suspicious_ports = [4444, 5555, 6667, 31337]  # Common backdoor/RAT ports

    num_malicious = int(num_packets * malicious_ratio)

    for i in range(num_packets):
        is_malicious = i < num_malicious

        if is_malicious:
            # Suspicious traffic patterns
            src_ip = np.random.choice(normal_ips)
            dst_ip = np.random.choice(suspicious_ips)
            dst_port = np.random.choice(suspicious_ports)
            size = np.random.randint(2000, 8000)  # Larger exfiltration packets
            protocol = 'TCP'

            # Add suspicious patterns
            if np.random.random() < 0.3:
                # Port scan pattern
                dst_port = np.random.randint(1, 65535)
                size = 64  # SYN scan packets are small
        else:
            # Normal traffic
            src_ip = np.random.choice(normal_ips)
            dst_ip = f"{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}"
            dst_port = np.random.choice(common_ports)
            size = np.random.randint(64, 1500)
            protocol = np.random.choice(['TCP', 'UDP', 'ICMP'], p=[0.7, 0.2, 0.1])

        packets.append({
            'timestamp': datetime.now().timestamp() + i * 0.01,
            'src_ip': src_ip,
            'dst_ip': dst_ip,
            'dst_port': dst_port,
            'protocol': protocol,
            'size': size,
            'malicious': is_malicious
        })

    return pd.DataFrame(packets)


def analyze_traffic(df):
    """Comprehensive traffic analysis"""

    print("═" * 70)
    print("📊 NETWORK TRAFFIC ANALYSIS REPORT")
    print("═" * 70)

    # Basic stats
    print(f"\n📦 Total Packets: {len(df):,}")
    print(f"📏 Total Data: {df['size'].sum() / 1024 / 1024:.2f} MB")
    print(f"⏱️  Time Span: {df['timestamp'].max() - df['timestamp'].min():.2f} seconds")

    # Protocol distribution
    print(f"\n🔌 Protocol Distribution:")
    for proto, count in df['protocol'].value_counts().items():
        print(f"   └─ {proto}: {count:,} packets ({count/len(df)*100:.1f}%)")

    # Top talkers
    print(f"\n💬 Top Source IPs:")
    for ip, count in df['src_ip'].value_counts().head(5).items():
        print(f"   └─ {ip}: {count:,} packets")

    # Suspicious ports
    suspicious_ports = df[df['dst_port'].isin([4444, 5555, 6667, 31337, 1337])]
    if len(suspicious_ports) > 0:
        print(f"\n⚠️  ALERT: Traffic detected on suspicious ports!")
        for port, count in suspicious_ports['dst_port'].value_counts().items():
            print(f"   └─ Port {port}: {count} connections (commonly used for C2/backdoors)")

    # Large packets (potential exfiltration)
    large_packets = df[df['size'] > 4000]
    if len(large_packets) > 0:
        print(f"\n📤 Large Outbound Packets Detected:")
        print(f"   └─ {len(large_packets)} packets > 4KB (potential data exfiltration)")
        print(f"   └─ Total: {large_packets['size'].sum() / 1024:.1f} KB")

    # Port scan detection
    port_scan = df.groupby('src_ip')['dst_port'].nunique()
    scanners = port_scan[port_scan > 50]
    if len(scanners) > 0:
        print(f"\n🎯 PORT SCAN DETECTED:")
        for ip, port_count in scanners.items():
            print(f"   └─ {ip} contacted {port_count} different ports (likely scanning)")

    print("\n" + "═" * 70)


def detect_anomalies(df):
    """
    Use Isolation Forest to detect anomalous network behavior
    """
    print("\n🧠 Training ML anomaly detection model...\n")

    # Feature engineering
    features = df[['dst_port', 'size']].copy()
    features['protocol_tcp'] = (df['protocol'] == 'TCP').astype(int)
    features['protocol_udp'] = (df['protocol'] == 'UDP').astype(int)

    # Normalize
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Train Isolation Forest
    model = IsolationForest(
        contamination=0.15,  # Expected proportion of anomalies
        random_state=42,
        n_estimators=100
    )

    predictions = model.fit_predict(features_scaled)
    df['ml_anomaly'] = predictions == -1  # -1 indicates anomaly

    # Results
    num_anomalies = df['ml_anomaly'].sum()
    true_positives = (df['ml_anomaly'] & df['malicious']).sum()

    print(f"✅ Model trained")
    print(f"\n📊 Results:")
    print(f"   └─ Flagged as anomalous: {num_anomalies} packets ({num_anomalies/len(df)*100:.1f}%)")
    print(f"   └─ True malicious detected: {true_positives} / {df['malicious'].sum()}")
    print(f"   └─ Detection rate: {true_positives/df['malicious'].sum()*100:.1f}%")

    # Show some anomalies
    print(f"\n⚠️  Sample Anomalies:")
    anomalies = df[df['ml_anomaly']].head(5)
    for idx, row in anomalies.iterrows():
        print(f"   └─ {row['src_ip']} → {row['dst_ip']}:{row['dst_port']} ({row['protocol']}, {row['size']} bytes)")

    return df


# ═══════════════════════════════════════════════════════════════════════════
# PART 2: VULNERABILITY SCANNING
# ═══════════════════════════════════════════════════════════════════════════

def simulate_scan_results():
    """
    Simulated scan results for demonstration.
    In production, you'd scan real targets with nmap.
    """
    return {
        '192.168.1.10': {
            'hostname': 'web-server-01',
            'state': 'up',
            'ports': [
                {'port': 22, 'state': 'open', 'service': 'ssh', 'version': 'OpenSSH 7.4'},
                {'port': 80, 'state': 'open', 'service': 'http', 'version': 'Apache 2.4.6'},
                {'port': 443, 'state': 'open', 'service': 'https', 'version': 'Apache 2.4.6'},
                {'port': 3306, 'state': 'open', 'service': 'mysql', 'version': 'MySQL 5.7.31'},
            ]
        },
        '192.168.1.11': {
            'hostname': 'file-server',
            'state': 'up',
            'ports': [
                {'port': 21, 'state': 'open', 'service': 'ftp', 'version': 'vsftpd 3.0.2'},
                {'port': 22, 'state': 'open', 'service': 'ssh', 'version': 'OpenSSH 8.2'},
                {'port': 445, 'state': 'open', 'service': 'smb', 'version': 'Samba 4.11.2'},
            ]
        },
        '192.168.1.12': {
            'hostname': 'dev-workstation',
            'state': 'up',
            'ports': [
                {'port': 22, 'state': 'open', 'service': 'ssh', 'version': 'OpenSSH 9.0'},
                {'port': 3389, 'state': 'open', 'service': 'rdp', 'version': 'Microsoft RDP'},
                {'port': 5900, 'state': 'open', 'service': 'vnc', 'version': 'VNC 4.1.2'},
            ]
        }
    }


def check_cve_database(service, version):
    """
    Simulate CVE lookup. In production, query NVD API or CVE databases.
    """
    vuln_db = {
        'Apache 2.4.6': [
            {'cve': 'CVE-2021-44790', 'severity': 'CRITICAL', 'cvss': 9.8,
             'description': 'Buffer overflow in mod_lua multipart parser'},
            {'cve': 'CVE-2021-33193', 'severity': 'HIGH', 'cvss': 7.5,
             'description': 'Request splitting via HTTP/2 method injection'}
        ],
        'OpenSSH 7.4': [
            {'cve': 'CVE-2018-15473', 'severity': 'MEDIUM', 'cvss': 5.3,
             'description': 'Username enumeration via malformed packets'}
        ],
        'MySQL 5.7.31': [
            {'cve': 'CVE-2021-2154', 'severity': 'HIGH', 'cvss': 7.1,
             'description': 'Server: DML unspecified vulnerability'},
            {'cve': 'CVE-2021-2166', 'severity': 'MEDIUM', 'cvss': 4.9,
             'description': 'Server: DML privilege escalation'}
        ],
        'vsftpd 3.0.2': [
            {'cve': 'CVE-2011-2523', 'severity': 'CRITICAL', 'cvss': 10.0,
             'description': 'Backdoor in vsftpd 3.0.2 (smiley backdoor)'}
        ],
        'VNC 4.1.2': [
            {'cve': 'CVE-2020-14401', 'severity': 'CRITICAL', 'cvss': 9.8,
             'description': 'Buffer overflow in libvncclient'}
        ]
    }

    return vuln_db.get(version, [])


def vulnerability_scan():
    """Run vulnerability scan and CVE lookup"""

    print("\n🔍 Starting network reconnaissance...\n")

    scan_results = simulate_scan_results()

    print("═" * 70)
    print("🎯 VULNERABILITY SCAN REPORT")
    print("═" * 70)

    for ip, data in scan_results.items():
        print(f"\n📍 Host: {ip} ({data['hostname']})")
        print(f"   Status: {data['state'].upper()}")
        print(f"   Open Ports:")

        for port_info in data['ports']:
            risk_indicator = ""
            if port_info['port'] in [21, 23, 445, 3389, 5900]:  # High-risk services
                risk_indicator = " ⚠️  HIGH RISK"
            elif port_info['port'] == 3306:  # MySQL exposed
                risk_indicator = " ⚡ EXPOSED DB"

            print(f"      └─ {port_info['port']}/{port_info['service']} - {port_info['version']}{risk_indicator}")

    print("\n" + "═" * 70)

    # CVE Assessment
    print("\n🔎 Checking for known CVEs...\n")
    print("═" * 70)
    print("🚨 VULNERABILITY ASSESSMENT REPORT")
    print("═" * 70)

    total_vulns = 0
    critical_count = 0
    high_count = 0
    vuln_data = []

    for ip, data in scan_results.items():
        host_vulns = []

        for port_info in data['ports']:
            cves = check_cve_database(port_info['service'], port_info['version'])
            if cves:
                host_vulns.extend([(port_info, cve) for cve in cves])
                for cve in cves:
                    vuln_data.append({
                        'host': ip,
                        'hostname': data['hostname'],
                        'service': port_info['service'],
                        'port': port_info['port'],
                        'cve': cve['cve'],
                        'severity': cve['severity'],
                        'cvss': cve['cvss']
                    })

        if host_vulns:
            print(f"\n🖥️  {ip} ({data['hostname']})")
            print(f"   └─ {len(host_vulns)} vulnerabilities found\n")

            for port_info, cve in host_vulns:
                severity_color = {
                    'CRITICAL': '🔴',
                    'HIGH': '🟠',
                    'MEDIUM': '🟡',
                    'LOW': '🟢'
                }

                icon = severity_color.get(cve['severity'], '⚪')

                print(f"   {icon} {cve['severity']} - {cve['cve']} (CVSS {cve['cvss']})")
                print(f"      Service: {port_info['service']} v{port_info['version']} on port {port_info['port']}")
                print(f"      Description: {cve['description']}")
                print()

                total_vulns += 1
                if cve['severity'] == 'CRITICAL':
                    critical_count += 1
                elif cve['severity'] == 'HIGH':
                    high_count += 1

    print("═" * 70)
    print(f"\n📊 Summary:")
    print(f"   └─ Total vulnerabilities: {total_vulns}")
    print(f"   └─ 🔴 Critical: {critical_count}")
    print(f"   └─ 🟠 High: {high_count}")
    print(f"\n⚠️  IMMEDIATE ACTION REQUIRED: Patch critical vulnerabilities ASAP!")

    return scan_results, pd.DataFrame(vuln_data)


# ═══════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════════════

def main():
    """Run the complete cybersecurity analysis"""

    print("═" * 70)
    print("🔒 CyberSec Lab: Network Traffic Analysis + Vulnerability Scanning")
    print("═" * 70)
    print()

    # Part 1: Network Traffic Analysis
    print("📊 PART 1: NETWORK TRAFFIC ANALYSIS")
    print("─" * 70)

    df_traffic = generate_sample_traffic(num_packets=1000, malicious_ratio=0.12)
    print(f"✅ Generated {len(df_traffic)} packets")
    print(f"   └─ {df_traffic['malicious'].sum()} suspicious packets ({df_traffic['malicious'].mean()*100:.1f}%)\n")

    analyze_traffic(df_traffic)
    df_traffic = detect_anomalies(df_traffic)

    # Part 2: Vulnerability Scanning
    print("\n\n🎯 PART 2: VULNERABILITY SCANNING")
    print("─" * 70)

    scan_results, df_vulns = vulnerability_scan()

    # Export reports
    print("\n\n📁 EXPORTING REPORTS")
    print("─" * 70)

    report = {
        'timestamp': datetime.now().isoformat(),
        'traffic_analysis': {
            'total_packets': len(df_traffic),
            'suspicious_packets': int(df_traffic['malicious'].sum()),
            'ml_detected_anomalies': int(df_traffic['ml_anomaly'].sum()),
            'protocols': df_traffic['protocol'].value_counts().to_dict()
        },
        'vulnerability_scan': {
            'hosts_scanned': len(scan_results),
            'total_vulnerabilities': len(df_vulns),
            'severity_breakdown': df_vulns['severity'].value_counts().to_dict(),
            'critical_cves': df_vulns[df_vulns['severity'] == 'CRITICAL']['cve'].tolist()
        }
    }

    with open('security_report.json', 'w') as f:
        json.dump(report, f, indent=2)

    df_traffic.to_csv('traffic_analysis.csv', index=False)
    df_vulns.to_csv('vulnerability_scan.csv', index=False)

    print("✅ Reports saved:")
    print("   └─ security_report.json")
    print("   └─ traffic_analysis.csv")
    print("   └─ vulnerability_scan.csv")

    print("\n" + "═" * 70)
    print("✅ Analysis complete!")
    print("═" * 70)


if __name__ == "__main__":
    main()